# 경기도 카드 소비 데이터 매출 예측 모델 학습 (전체 데이터)

## 사전 준비
1. 캐글 데이터셋에 `dataset/` 폴더의 CSV 파일들을 업로드
2. 이 노트북에서 해당 데이터셋을 추가 (우측 Data 탭)
3. Session Options → Accelerator: **GPU P100** 또는 **CPU** 선택
4. `Run All` 실행 후 Output 탭에서 모델 파일 다운로드

In [ ]:
import os, glob, time
import numpy as np
import pandas as pd
import joblib
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder

# ── 경로 설정 ──────────────────────────────────────────────
# 캐글 데이터셋 추가 후 실제 경로로 수정
# 예: /kaggle/input/[데이터셋-이름]/
DATA_DIR   = "/kaggle/input/gyeonggi-card-data"
OUTPUT_DIR = "/kaggle/working"

os.makedirs(f"{OUTPUT_DIR}/model",    exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/encoders", exist_ok=True)

files = sorted(glob.glob(f"{DATA_DIR}/tbsh_gyeonggi_day_*.csv"))
print(f"파일 수: {len(files)}개")
for f in files:
    print(f"  {os.path.basename(f)}")

In [ ]:
# =========================================================
# 설정
# =========================================================
MODEL_FEATURES = ["sex", "age", "day", "hour", "month",
                  "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2", "cnt"]
LABEL_COLS  = ["age", "day", "hour", "month"]
ONEHOT_COLS = ["sex", "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2"]
NUMERIC_COLS = ["cnt"]

REMOVE_OUTLIER = True

# dtype 최적화 (메모리 절약)
DTYPES = {
    "admi_cty_no":      "int32",
    "hour":             "int8",
    "sex":              "category",
    "age":              "int8",
    "day":              "int8",
    "amt":              "int64",
    "cnt":              "int32",
    "card_tpbuz_nm_1":  "category",
    "card_tpbuz_nm_2":  "category",
}

In [ ]:
# =========================================================
# 1) 전체 데이터 로드 (dtype 최적화)
# =========================================================
dfs = []
t0 = time.time()

for i, f in enumerate(files, 1):
    city = os.path.basename(f).split("_")[-1].replace(".csv", "")
    df = pd.read_csv(
        f, encoding="utf-8-sig",
        usecols=["ta_ymd", "admi_cty_no", "card_tpbuz_nm_1", "card_tpbuz_nm_2",
                 "hour", "sex", "age", "day", "amt", "cnt"],
        dtype=DTYPES
    )
    df["month"] = df["ta_ymd"].astype(str).str[4:6].astype("int8")
    df["admi_cty_no"] = df["admi_cty_no"].astype(str).astype("category")
    df["log_amt"] = np.log1p(df["amt"]).astype("float32")

    if REMOVE_OUTLIER:
        upper = df["amt"].quantile(0.99)
        df = df[df["amt"] <= upper]

    dfs.append(df[MODEL_FEATURES + ["log_amt"]].dropna())
    print(f"[{i}/{len(files)}] {city}: {len(dfs[-1]):,}행  ({time.time()-t0:.0f}s 경과)")

df_all = pd.concat(dfs, ignore_index=True)
del dfs
import gc; gc.collect()

print(f"\n전체 학습 데이터: {len(df_all):,}행")
print(f"메모리 사용량: {df_all.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# =========================================================
# 2) 인코딩
# =========================================================
X = df_all[MODEL_FEATURES].copy()
y = df_all["log_amt"].astype("float32")
del df_all; gc.collect()

for col in LABEL_COLS:
    X[col] = X[col].astype(str)

label_encoders = {}
for col in LABEL_COLS:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    label_encoders[col] = le

ohe = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_arr   = ohe.fit_transform(X[ONEHOT_COLS].astype(str))
ohe_names = ohe.get_feature_names_out(ONEHOT_COLS).tolist()
ohe_df    = pd.DataFrame(ohe_arr, columns=ohe_names, index=X.index)
del ohe_arr; gc.collect()

encoded_X = pd.concat([
    X[LABEL_COLS + NUMERIC_COLS].reset_index(drop=True),
    ohe_df.reset_index(drop=True)
], axis=1)
feature_columns = encoded_X.columns.tolist()
del X, ohe_df; gc.collect()

print(f"피처 수: {len(feature_columns)}개")
print(f"인코딩 후 메모리: {encoded_X.memory_usage(deep=True).sum() / 1e9:.2f} GB")

In [ ]:
# =========================================================
# 3) 학습 / 검증 분할
# =========================================================
X_train, X_test, y_train, y_test = train_test_split(
    encoded_X, y, test_size=0.2, random_state=42)
del encoded_X, y; gc.collect()

print(f"학습: {len(X_train):,}행 / 검증: {len(X_test):,}행")

In [ ]:
# =========================================================
# 4) LightGBM 학습
# =========================================================
t0 = time.time()

model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
    device="gpu"       # GPU 없으면 "cpu"로 자동 fallback
)

try:
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(100)
        ]
    )
except Exception as e:
    # GPU 실패 시 CPU로 재시도
    print(f"GPU 학습 실패 ({e}), CPU로 재시도합니다.")
    model.set_params(device="cpu")
    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(100)
        ]
    )

print(f"\n학습 완료: {time.time()-t0:.1f}초")

In [ ]:
# =========================================================
# 5) 평가
# =========================================================
pred = model.predict(X_test)
y_real    = np.expm1(y_test)
pred_real = np.maximum(np.expm1(pred), 0)

r2   = r2_score(y_real, pred_real)
rmse = np.sqrt(mean_squared_error(y_real, pred_real))
mae  = mean_absolute_error(y_real, pred_real)

print(f"R²   = {r2:.4f}")
print(f"RMSE = {rmse:,.0f}")
print(f"MAE  = {mae:,.0f}")

In [ ]:
# =========================================================
# 6) 저장
# =========================================================
joblib.dump(model,           f"{OUTPUT_DIR}/model/sales_predict_model.pkl")
joblib.dump(label_encoders,  f"{OUTPUT_DIR}/encoders/label_encoders.pkl")
joblib.dump(ohe,             f"{OUTPUT_DIR}/encoders/onehot_encoder.pkl")
joblib.dump(feature_columns, f"{OUTPUT_DIR}/encoders/feature_columns.pkl")
joblib.dump({
    "model_name":      "LightGBM",
    "R2":              r2,
    "RMSE":            rmse,
    "MAE":             mae,
    "total_rows":      len(X_train) + len(X_test),
    "features":        MODEL_FEATURES,
    "use_log_target":  True,
    "remove_outliers": REMOVE_OUTLIER,
}, f"{OUTPUT_DIR}/model/model_info.pkl")

print("저장 완료!")
print("\n오른쪽 Output 탭에서 아래 파일들을 다운로드하세요:")
print("  model/sales_predict_model.pkl")
print("  model/model_info.pkl")
print("  encoders/label_encoders.pkl")
print("  encoders/onehot_encoder.pkl")
print("  encoders/feature_columns.pkl")